# Qwen3.8-27B Uncensored (HauhauCS Aggressive) — Q3_K_P di Kaggle

Notebook ini: download GGUF Q3_K_P (13,44 GB) → build `llama.cpp` CUDA → jalanin `llama-server` → chat lewat Gradio.

**Setting Kaggle sebelum Run (penting):**
- Kanan atas / Settings → Accelerator → **GPU T4 x2** (jangan P100, jangan None)
- Settings → Internet → **ON** (buat download model dari Hugging Face)
- Akun Kaggle harus sudah verifikasi nomor HP, kalau tidak GPU + Internet tidak kebuka.

**Jalankan cell berurutan 1 → 6. Jangan loncat.** Cell 6 (Gradio) itu blocking — selama cell itu jalan, server hidup. Kalau cell 6 di-stop, link publik ikut mati.

**Estimasi run pertama:** download ~10–20 menit (tergantung speed Kaggle↔HF) + build llama.cpp ~10–25 menit. Run kedua di sesi baru tetap download ulang kecuali model sudah disimpan jadi Kaggle Dataset (lihat catatan di akhir).

**Batasan Kaggle:** sesi maks ~12 jam, kuota ~30 GPU-jam/minggu, `/kaggle/working` max ~20 GB. Notebook ini bukan server 24/7.

**Keamanan — baca dulu:** link Gradio `https://....gradio.live` itu PUBLIK. Siapa pun yang pegang link bisa pakai GPU-mu. Model ini uncensored (0 refusal). Jangan share link ke publik / grup. Kalau mau aman, isi `GRADIO_PASSWORD` di Cell 1, atau pakai Cloudflare Tunnel + Access untuk API (di luar notebook ini).

In [ ]:
# CELL 1 — Config. Edit di sini saja kalau mau ubah.
import os
import pathlib

MODEL_REPO = "HauhauCS/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF"
MODEL_FILE = "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-Q3_K_P.gguf"
EXPECTED_GB = 13.44  # dari halaman HF, buat cek kasar saja
CTX = 16384  # jangan naikkan ke 262144 di T4, KV cache akan OOM. 8192 kalau masih OOM.
LLAMA_PORT = 8080
GRADIO_PASSWORD = ""  # isi, misal "rahasia123", biar Gradio minta login user/pass. Kosong = tanpa login (tidak disarankan).
ENABLE_THINKING = False  # False = jawaban langsung, True = model mikir dulu pakai <think>

# Path: Kaggle pakai /kaggle/working, lokal pakai ./
BASE = pathlib.Path("/tmp/qwen38-run")  # push/batch: /tmp besar, /kaggle/working cuma ~20GB (model 13.44GB + build llama.cpp mepet)
BASE.mkdir(parents=True, exist_ok=True)
MODEL_DIR = BASE / "models"
MODEL_PATH = MODEL_DIR / MODEL_FILE
LLAMA_DIR = BASE / "llama.cpp"
LLAMA_SERVER = LLAMA_DIR / "build" / "bin" / "llama-server"

print(f"BASE: {BASE}")
print(f"MODEL_PATH: {MODEL_PATH}")
print(f"LLAMA_SERVER: {LLAMA_SERVER}")
print(f"CTX: {CTX} | ENABLE_THINKING: {ENABLE_THINKING}")
print(f"GRADIO_PASSWORD terisi: {bool(GRADIO_PASSWORD)}")

In [ ]:
# CELL 2 — Cek GPU. Harus muncul 2x Tesla T4.
import subprocess
import shutil

print(shutil.which("nvidia-smi"))
r = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv"], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
print("Jika di atas TIDAK muncul 2x T4: stop, ganti Accelerator ke GPU T4 x2, lalu restart session.")

In [ ]:
# CELL 3 — Install deps Python (tidak install llama.cpp di sini, itu di Cell 5).
import subprocess
import sys

pkgs = ["huggingface_hub", "hf_transfer", "gradio", "openai", "requests"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs, text=True, capture_output=True)
print(r.stdout[-2000:] if r.stdout else "")
print(r.stderr[-2000:] if r.stderr else "")
print("returncode:", r.returncode)

In [ ]:
# CELL 4 — Download model dari Hugging Face.
# Prioritas: kalau file sudah ada di /kaggle/input (dari Dataset), pakai itu. Kalau tidak, download ke /kaggle/working/models.
import glob
import os
import pathlib

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# 1) Cek Dataset input Kaggle
found = glob.glob(f"/kaggle/input/**/{MODEL_FILE}", recursive=True)
if found:
    print(f"Ketemu di /kaggle/input: {found[0]}")
    MODEL_PATH = pathlib.Path(found[0])
    print(f"Pakai MODEL_PATH = {MODEL_PATH}")
else:
    print("Tidak ada di /kaggle/input, download dari HF...")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    # Kalau file sudah ada + ukuran masuk akal, skip download
    if MODEL_PATH.exists() and MODEL_PATH.stat().st_size > 10_000_000_000:
        print(f"Sudah ada, skip download: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e9:.2f} GB)")
    else:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=str(MODEL_DIR), local_dir_use_symlinks=False, resume_download=True)
        print(f"Download selesai: {p}")
        MODEL_PATH = pathlib.Path(p)

# 2) Verifikasi ukuran
size_gb = MODEL_PATH.stat().st_size / 1e9
print(f"File: {MODEL_PATH}")
print(f"Ukuran: {size_gb:.2f} GB (ekspektasi ~{EXPECTED_GB} GB)")
if size_gb < 10:
    raise RuntimeError("File terlalu kecil, download belum lengkap / corrupt. Hapus file dan ulangi Cell 4.")
print("OK — lanjut ke Cell 5.")

In [ ]:
# CELL 5 — Ambil llama-server PREBUILT via URL pinned (tanpa panggil API GitHub).
# v2 gagal: API GitHub -> RemoteDisconnected dari Kaggle.
# v1 gagal: build source -> cmake CUDA::cuda_driver not found.
# Jadi pakai URL release b11401 CUDA 12.8 langsung (terverifikasi ada), download pakai curl --retry.
import os
import shutil
import subprocess
import tarfile

PINNED_URL = "https://github.com/ggml-org/llama.cpp/releases/download/b11401/llama-b11401-bin-ubuntu-cuda-12.8-x64.tar.gz"
CUDART_URL = "https://github.com/ggml-org/llama.cpp/releases/download/b11401/cudart-llama-b11401-bin-ubuntu-cuda-12.8-x64.tar.gz"

if LLAMA_SERVER.exists():
    print(f"Sudah ada, skip download: {LLAMA_SERVER}")
else:
    extract_dir = BASE / "llama-extract"
    extract_dir.mkdir(parents=True, exist_ok=True)
    for url, name in [(PINNED_URL, "llama"), (CUDART_URL, "cudart")]:
        tgz = BASE / f"{name}-b11401.tar.gz"
        print(f"Download {name}: {url}")
        r = subprocess.run(["curl", "-L", "--retry", "3", "--retry-delay", "5", "--fail", "-o", str(tgz), url], text=True, capture_output=True)
        print(r.stdout[-1000:] if r.stdout else "")
        print(r.stderr[-1000:] if r.stderr else "")
        if r.returncode != 0:
            raise RuntimeError(f"curl download {name} gagal (code {r.returncode})")
        print(f"Download {name} selesai: {tgz.stat().st_size / 1e6:.1f} MB, extract...")
        with tarfile.open(str(tgz)) as tf:
            tf.extractall(path=str(extract_dir), filter="data")
    hits = list(extract_dir.rglob("llama-server"))
    assert hits, "llama-server tidak ketemu di dalam release tarball"
    src_bin = hits[0]
    print(f"Ketemu binary: {src_bin}")
    LLAMA_SERVER.parent.mkdir(parents=True, exist_ok=True)
    # JANGAN cuma copy binary-nya: .so sibling (libllama-server-impl.so dll) harus ikut.
    # v2/v3 gagal di sini — binary dicopy sendirian, .so ketinggalan di extract dir.
    for item in src_bin.parent.iterdir():
        dest = LLAMA_SERVER.parent / item.name
        if item.is_dir():
            shutil.copytree(str(item), str(dest), dirs_exist_ok=True)
        else:
            shutil.copy(str(item), str(dest))
    LLAMA_SERVER.chmod(0o755)
    print(f"Disalin se-direktori ke: {LLAMA_SERVER.parent} (binary + .so sibling)")
    # Biar llama-server ketemu .so CUDA dari tarball, tambahkan lib dir ke LD_LIBRARY_PATH (berlaku untuk Cell 6a, kernel yang sama)
    lib_dirs = [str(LLAMA_SERVER.parent)] + [str(p) for p in extract_dir.rglob("lib") if p.is_dir()]
    if lib_dirs:
        old = os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + ([old] if old else []))
        print(f"LD_LIBRARY_PATH += {lib_dirs}")

print(f"Cek binary: {LLAMA_SERVER} exists={LLAMA_SERVER.exists()}")
r = subprocess.run([str(LLAMA_SERVER), "--version"], capture_output=True, text=True, env=dict(os.environ))
print(r.stdout)
print(r.stderr)
if r.returncode != 0:
    raise RuntimeError("llama-server --version gagal — binary prebuilt tidak jalan di image ini")


In [ ]:
# CELL 6a — Start llama-server di background + tunggu sampai /health OK.
# Jangan run cell ini 2x tanpa stop server lama (port 8080 bentrok).
import json
import subprocess
import time
import urllib.request

# Bunuh server lama di port ini kalau ada (aman di Kaggle, sesi milik sendiri)
subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)
time.sleep(2)

# Simpan path model yang dipakai Cell 4 (bisa dari /kaggle/input)
model_arg = str(MODEL_PATH)
print(f"Model: {model_arg}")

cmd = [
    str(LLAMA_SERVER),
    "--model", model_arg,
    "--ctx-size", str(CTX),
    "--n-gpu-layers", "999",
    "--tensor-split", "1,1",
    "--split-mode", "layer",
    "--jinja",
    "--host", "127.0.0.1",
    "--port", str(LLAMA_PORT),
    "--parallel", "1",
    "--batch-size", "512",
    "--ubatch-size", "256",
    "--temp", "0.7",
    "--top-k", "20",
    "--top-p", "0.8",
    "--min-p", "0.0",
    "--repeat-penalty", "1.0",
    "--chat-template-kwargs", json.dumps({"enable_thinking": ENABLE_THINKING}),
]
print("CMD:", " ".join(cmd))
logf = open(str(BASE / "llama-server.log"), "w")
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
print(f"llama-server PID: {proc.pid}, log: {BASE / 'llama-server.log'}")

# Tunggu health sampai 10 menit
ok = False
for i in range(120):
    if proc.poll() is not None:
        print(f"Server exit code {proc.returncode}, cek log:")
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{LLAMA_PORT}/health", timeout=5) as resp:
            body = resp.read().decode()
            print(f"health ({i*5}s): {body[:500]}")
            if resp.status == 200:
                ok = True
                break
    except Exception as e:
        if i % 12 == 0:
            print(f"menunggu... {i*5}s ({e})")
    time.sleep(5)

if not ok:
    print("Server belum OK. Buka log 100 baris terakhir:")
    print(open(str(BASE / "llama-server.log")).read()[-5000:])
    raise RuntimeError("llama-server gagal start — cek log di atas")

# OFFLOAD DIAG — buktikan model jalan di GPU, bukan CPU (v3 smoke test dicurigai lambat ~0.08 tok/s)
print("--- nvidia-smi setelah server start ---")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu", "--format=csv"], capture_output=True, text=True).stdout)
print("--- llama-server.log tail (offload / CUDA) ---")
try:
    logtxt = open(str(BASE / "llama-server.log")).read()
    for line in logtxt.splitlines():
        low = line.lower()
        if any(k in low for k in ["offload", "cuda", "gpu", "vram", "layers", "total", "memory"]):
            print(line[:400])
    print(f"(log total {len(logtxt.splitlines())} lines)")
except Exception as e:
    print(f"baca log gagal: {e}")

# Cek model id yang dilaporkan server (dipakai Gradio + API)
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{LLAMA_PORT}/v1/models", timeout=10) as resp:
        print("models:", resp.read().decode()[:1000])
except Exception as e:
    print(f"/v1/models gagal: {e}")

# Smoke test 1 token (timed — kalau <2 tok/s, curiga CPU offload gagal), biar ketahuan OOM / error sebelum buka Gradio
import time as _time
_t0 = _time.time()
try:
    import urllib.request as _u
    payload = json.dumps({"model": "default", "messages": [{"role": "user", "content": "Balas dengan kata: OK"}], "max_tokens": 5, "temperature": 0.0, "stream": False}).encode()
    req = _u.Request(f"http://127.0.0.1:{LLAMA_PORT}/v1/chat/completions", data=payload, headers={"Content-Type": "application/json"})
    with _u.urlopen(req, timeout=120) as resp:
        print("smoke test:", resp.read().decode()[:1000])
        print(f"smoke test elapsed: {_time.time() - _t0:.1f}s")
except Exception as e:
    print(f"smoke test gagal: {e} — cek log, kemungkinan OOM / model id salah")

print("OK — lanjut ke Cell 6b (Gradio).")

In [ ]:
# CELL 6b — Chat UI Gradio (BLOCKING). Biarkan cell ini jalan.
# Link publik muncul di output Gradio: https://....gradio.live
# API: <link>/v1/chat/completions (OpenAI-compatible). Cek model id di /v1/models.
# Format history: Gradio di image ini pakai MESSAGES format (list dict {role, content}).
# v8 gagal di sini: chat_fn yield pasangan [user, assistant] (tuples) -> Gradio tolak
# "Data incompatible with messages format". Jadi chat_fn sekarang yield list dict messages.
# Chatbot polos (tanpa type=) — Gradio lama tidak kenal argumen type= (v4 gagal di situ).
import gradio as gr
import json
import urllib.request

# Ambil model id asli dari server, fallback ke 'default'
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{LLAMA_PORT}/v1/models", timeout=10) as resp:
        MODEL_ID = json.loads(resp.read().decode())["data"][0]["id"]
except Exception:
    MODEL_ID = "default"
print(f"MODEL_ID untuk API: {MODEL_ID}")

from openai import OpenAI
client = OpenAI(base_url=f"http://127.0.0.1:{LLAMA_PORT}/v1", api_key="not-used")

def chat_fn(message, history, temperature, max_tokens):
    if not message:
        yield history or [], ""
        return
    # Normalisasi history ke messages-format dicts (Gradio ini pakai messages, bukan tuples)
    msgs = []
    for h in (history or []):
        if isinstance(h, dict) and "role" in h and "content" in h:
            msgs.append({"role": h["role"], "content": h["content"]})
        elif isinstance(h, (list, tuple)) and len(h) == 2:
            if h[0]:
                msgs.append({"role": "user", "content": h[0]})
            if h[1]:
                msgs.append({"role": "assistant", "content": h[1]})
    msgs.append({"role": "user", "content": message})
    out = ""
    # enable_thinking sudah diset di server (Cell 6a --chat-template-kwargs)
    stream = client.chat.completions.create(model=MODEL_ID, messages=msgs, temperature=float(temperature), max_tokens=int(max_tokens), stream=True)
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        out += delta
        yield msgs + [{"role": "assistant", "content": out}], ""

with gr.Blocks(title="Qwen3.8-27B Q3_K_P — Kaggle Chat") as demo:
    gr.Markdown("## Qwen3.8-27B Uncensored (HauhauCS Aggressive) — Q3_K_P @ Kaggle T4 x2\nJangan share link ini. Sesi Kaggle maks ~12 jam.")
    chatbot = gr.Chatbot(label="Chat")
    msg = gr.Textbox(label="Pesan", placeholder="Tulis pesan, Enter untuk kirim...")
    with gr.Row():
        temp = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label="Temperature")
        maxtok = gr.Slider(64, 4096, value=1024, step=64, label="Max tokens")
    with gr.Row():
        send = gr.Button("Kirim", variant="primary")
        clear = gr.Button("Clear")
    send.click(chat_fn, inputs=[msg, chatbot, temp, maxtok], outputs=[chatbot, msg])
    msg.submit(chat_fn, inputs=[msg, chatbot, temp, maxtok], outputs=[chatbot, msg])
    clear.click(lambda: ([], ""), None, [chatbot, msg])

auth = ("user", GRADIO_PASSWORD) if GRADIO_PASSWORD else None
if auth is None:
    print("WARNING: Gradio tanpa password — link publik bisa dipakai siapa pun yang pegang link.")
demo.launch(share=True, auth=auth, server_name="127.0.0.1", server_port=7860, prevent_thread_lock=True)
print("Gradio jalan. Keep-alive sampai sesi Kaggle berakhir (maks ~12 jam). Jangan stop cell ini.")
# KEEP-ALIVE — demo.launch() balik langsung di mode batch/papermill,
# notebook dianggap COMPLETE, kernel dimatikan, link Gradio jadi 404. Loop ini yang pegang kernel tetap RUNNING.
import time
while True:
    time.sleep(3600)


## Cara pakai dari laptop (API)

Setelah Cell 6b jalan, Gradio akan print link `https://....gradio.live`. API-nya di `<link>/v1`.

```python
from openai import OpenAI
client = OpenAI(base_url="https://XXXX.gradio.live/v1", api_key="not-used")
# model id = yang diprint di Cell 6a/6b dari /v1/models
r = client.chat.completions.create(model="<MODEL_ID>", messages=[{"role": "user", "content": "Halo"}])
print(r.choices[0].message.content)
```

## Biar tidak download ulang tiap sesi

1. Setelah Cell 4 selesai di sesi pertama, file model ada di `/kaggle/working/models/`.
2. Di panel kanan Kaggle → Output → download file-nya, atau Save Version → jadikan Output.
3. Untuk sesi berikutnya yang paling praktis: upload file `.gguf` itu sekali saja sebagai Kaggle Dataset privat (Datasets → New Dataset), lalu di notebook baru: Add Input → dataset itu. Cell 4 akan otomatis pakai file dari `/kaggle/input` tanpa download ulang.
4. Jangan upload `.gguf` ke repo GitHub — file 13 GB melebihi limit GitHub.

## Kalau OOM / lambat

- Turunkan `CTX` di Cell 1 ke `8192`, lalu restart dari Cell 6a (server lama di-`pkill` otomatis di Cell 6a).
- Cek `llama-server.log` di `/kaggle/working/llama-server.log` dan `nvidia-smi` di Cell 2. Kalau VRAM 2x T4 penuh tapi token/s sangat lambat, berarti ada layer yang jatuh ke CPU — biasanya karena `CTX` terlalu besar atau `tensor-split` salah.
- Jangan jalankan 2 notebook GPU bersamaan — kuota 30 jam/minggu dipakai bareng.